# 01 - Setup, data, preprocessing, EDA
Runs on Colab or Kaggle (GPU). Run top to bottom. Everything heavy is written to `BASE`, so point it at **Google Drive** (Colab) or `/kaggle/working` (Kaggle) to survive session death.

In [ ]:
# --- 0. Config -------------------------------------------------------------
import os, sys, subprocess
ON_COLAB = 'google.colab' in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['BASE'] = '/content/drive/MyDrive/brats_nnunet_data'
else:
    os.environ['BASE'] = '/kaggle/working/data'
BASE = os.environ['BASE']
os.makedirs(BASE, exist_ok=True)
print('BASE =', BASE)

In [ ]:
# --- 1. Get the repo + install deps (replace URL with your GitHub repo) -------
REPO_URL = 'https://github.com/Hidaayet/brats-nnunet'
if not os.path.exists('brats-nnunet'):
    !git clone $REPO_URL
%cd brats-nnunet
!pip install -q -r requirements.txt
!nvidia-smi | head -n 15

In [ ]:
# --- 2. nnU-Net env vars (must be set in-process AND for shell cells) ---------
os.environ['nnUNet_raw']          = f'{BASE}/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = f'{BASE}/nnUNet_preprocessed'
os.environ['nnUNet_results']      = f'{BASE}/nnUNet_results'
for k in ['nnUNet_raw','nnUNet_preprocessed','nnUNet_results']:
    os.makedirs(os.environ[k], exist_ok=True)

In [ ]:
# --- 3. Download MSD Task01 (~7 GB), convert, plan, preprocess (30-60 min) ---
!bash scripts/01_prepare_data.sh

## EDA: what are we segmenting?
Labels in MSD Task01: 1 = edema, 2 = non-enhancing tumor, 3 = enhancing tumor. Modalities: FLAIR, T1w, T1gd (post-contrast), T2w.

In [ ]:
import numpy as np, nibabel as nib, matplotlib.pyplot as plt, glob, pandas as pd
raw = f"{os.environ['nnUNet_raw']}/Dataset001_BrainTumour"
cases = sorted(os.path.basename(f)[:-7] for f in glob.glob(f'{raw}/labelsTr/*.nii.gz'))
print(len(cases), 'cases')

def show(cid, z=None):
    mods = ['FLAIR','T1w','T1gd','T2w']
    imgs = [np.asarray(nib.load(f'{raw}/imagesTr/{cid}_{m:04d}.nii.gz').dataobj) for m in range(4)]
    lab = np.asarray(nib.load(f'{raw}/labelsTr/{cid}.nii.gz').dataobj)
    if z is None: z = int(np.argmax((lab > 0).sum((0,1))))   # slice with most tumor
    fig, ax = plt.subplots(1, 5, figsize=(18, 3.8))
    for i, (im, n) in enumerate(zip(imgs, mods)):
        ax[i].imshow(np.rot90(im[..., z]), cmap='gray'); ax[i].set_title(n); ax[i].axis('off')
    ax[4].imshow(np.rot90(imgs[0][..., z]), cmap='gray')
    ax[4].imshow(np.ma.masked_equal(np.rot90(lab[..., z]), 0), cmap='autumn', alpha=.6, vmin=1, vmax=3)
    ax[4].set_title('Label overlay'); ax[4].axis('off')
    plt.tight_layout(); return fig

fig = show(cases[0]); fig.savefig('results/figures/eda_example.png', dpi=150)

In [ ]:
# Class balance + tumor volume per case (shows why Dice on small ET regions is hard)
rows = []
for c in cases[:120]:   # subset for speed; remove [:120] for the full set
    lab = np.asarray(nib.load(f'{raw}/labelsTr/{c}.nii.gz').dataobj)
    rows.append({'case': c, 'edema': (lab==1).sum(), 'non_enh': (lab==2).sum(), 'enh': (lab==3).sum(),
                 'brain_vox': lab.size})
df = pd.DataFrame(rows)
print(df[['edema','non_enh','enh']].describe().round(0))
print('cases with zero enhancing voxels:', int((df.enh==0).sum()))
ax = df[['edema','non_enh','enh']].plot.box(figsize=(6,4), logy=True, title='Voxels per class (log)')
plt.savefig('results/figures/eda_class_balance.png', dpi=150, bbox_inches='tight')

## Next: train
In a **new session**, re-run cells 0-2 (skip the download, the data is on Drive), then:
```
!bash scripts/02_train_nnunet.sh 3d_fullres 0     # main model
!bash scripts/02_train_nnunet.sh 2d 0             # 2D vs 3D comparison
!python -m src.baseline train --raw $nnUNet_raw/Dataset001_BrainTumour --epochs 20
```
If a session dies, rerun the same command: `--c` resumes from the last checkpoint.